In [3]:
import numpy as np
import pandas as pd
from scipy.special import spherical_jn
from scipy.integrate import cumulative_trapezoid, simpson

csv_file = "/root/geant4/fac/Kr_all_wavefunctions.csv"

df = pd.read_csv(csv_file)
df.columns = df.columns.str.strip()

# choose exact orbital from your output
orbital_name = "Kr_2p12"

# select orbital first
d = df[df["Orbital"].astype(str).str.strip() == orbital_name].copy()

# force r, P, Q to numbers
d["r"] = pd.to_numeric(d["r"], errors="coerce")
d["P"] = pd.to_numeric(d["P"], errors="coerce")
d["Q"] = pd.to_numeric(d["Q"], errors="coerce")

# remove bad rows
d = d.dropna(subset=["r", "P", "Q"])

print("Selected rows =", len(d))

r = d["r"].to_numpy(float)
P = d["P"].to_numpy(float)
Q = d["Q"].to_numpy(float)

idx = np.argsort(r)
r, P, Q = r[idx], P[idx], Q[idx]

print("r min/max =", r.min(), r.max())
print("P min/max =", P.min(), P.max())
print("Q min/max =", Q.min(), Q.max())

# 2p1/2
l_large = 1
l_small = 0

p_grid = np.arange(0.0, 300.0 + 0.05, 0.05)
const = np.sqrt(2/np.pi)

chiG = []
chiF = []

for p in p_grid:
   chi_g = const * simpson(P * spherical_jn(l_large, p*r) * r, x=r)
   chi_f = const * simpson(Q * spherical_jn(l_small, p*r) * r, x=r)

   chiG.append(chi_g)
   chiF.append(chi_f)

chiG = np.array(chiG)
chiF = np.array(chiF)
chiTotal = np.sqrt(chiG**2 + chiF**2)

I_p = chiTotal**2 * p_grid**2

J_integrand = chiTotal**2 * p_grid
J_Q = -0.5 * cumulative_trapezoid(
    J_integrand[::-1],
    p_grid[::-1],
    initial=0
)[::-1]

result = pd.DataFrame({
    "p": p_grid,
    "ChiF": chiF,
    "ChiG": chiG,
    "ChiTotal": chiTotal,
    "I(p)": I_p,
    "J(Q)": J_Q
})

print(result.head(40))

result.to_csv("Kr_simpson_2p12_integration_vs_p.csv", index=False)

Selected rows = 234
r min/max = 2.77777778e-08 2.39320801
P min/max = 2.034427e-11 2.449109
Q min/max = -0.01714749 0.2203886
       p          ChiF      ChiG  ChiTotal          I(p)      J(Q)
0   0.00 -1.151466e-05  0.000000  0.000012  0.000000e+00  0.043404
1   0.05 -1.143931e-05  0.000413  0.000413  4.257179e-10  0.043404
2   0.10 -1.121330e-05  0.000825  0.000825  6.806024e-09  0.043404
3   0.15 -1.083672e-05  0.001237  0.001237  3.443924e-08  0.043404
4   0.20 -1.030975e-05  0.001649  0.001649  1.087855e-07  0.043404
5   0.25 -9.632599e-06  0.002061  0.002061  2.654112e-07  0.043404
6   0.30 -8.805576e-06  0.002472  0.002472  5.499115e-07  0.043404
7   0.35 -7.829037e-06  0.002882  0.002882  1.017811e-06  0.043404
8   0.40 -6.703408e-06  0.003292  0.003292  1.734442e-06  0.043404
9   0.45 -5.429179e-06  0.003702  0.003702  2.774805e-06  0.043404
10  0.50 -4.006904e-06  0.004110  0.004110  4.223401e-06  0.043404
11  0.55 -2.437199e-06  0.004518  0.004518  6.174052e-06  0.043404
12 